# Fine-tuning Models on Synthetically Generated Data
## QLoRA Fine-tuning for Emotional Understanding & Application

This notebook fine-tunes GPT-OSS-20B using QLoRA (4-bit quantization + LoRA) on synthetic EmoBench-style EU and EA data.

**Pipeline:**
1. Load and convert EU/EA data to instruction-response format
2. Prepare training/validation splits
3. Load GPT-OSS-20B with 4-bit quantization
4. Apply LoRA adapters
5. Fine-tune with instruction-following format
6. Evaluate on real EmoBench data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json

INPUT_FILE = "/content/drive/MyDrive/685_Project_New/input_data/finetuning/Arjun/eu_ea_dataset_merged.json"      # your file
OUTPUT_FILE = "/content/drive/MyDrive/685_Project_New/input_data/finetuning/Arjun/arjuns_generation.jsonl"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)   # list of persona objects

items_only = []

for persona in data:
    persona_items = persona.get("items", [])
    for item in persona_items:
        items_only.append(item)



with open("/content/drive/MyDrive/685_Project_New/input_data/finetuning/Arjun/eu_ea_dataset_merged_list.json", "r", encoding="utf-8") as f:
    data = json.load(f)   # list of persona objects
print(len(data))
print(items_only)
total_data=items_only+data
print(f"Extracted {len(total_data)} items")
# Save as JSONL (best for HF datasets)
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in total_data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved to {OUTPUT_FILE}")


2392
[{'id': 'EU001', 'task_type': 'EU', 'scenario': 'The client reports that he has been worrying about how the disagreement might damage his reputation and friendships, and this worry keeps him up at night and makes him uneasy in social situations.', 'question': 'Which emotion is most likely present when the client thinks about the conflict?', 'choices': {'A': 'Anger', 'B': 'Anxiety', 'C': 'Contentment', 'D': 'Excitement'}, 'answer': 'B', 'rationale': 'The client’s description of worry, sleeplessness, and unease in social settings indicates anxiety rather than other emotions.'}, {'id': 'EU002', 'task_type': 'EU', 'scenario': 'When imagining a conversation that could lead to a positive outcome, the client says he feels hopeful, calmer, with a touch of nervousness that feels more like excitement than worry.', 'question': 'Which emotion best describes the client’s feeling when he imagines a positive conversation?', 'choices': {'A': 'Hopeful', 'B': 'Anxious', 'C': 'Angry', 'D': 'Sad'}, '

In [ ]:
import jsonlines

file1 = "ea_items.jsonl"
file2 = "ea_items-2.jsonl"
output = "ea_items_merged.jsonl"

with jsonlines.open(output, "w") as writer:
    # Write all entries from file1
    with jsonlines.open(file1, "r") as reader1:
        for obj in reader1:
            writer.write(obj)

    # Append all entries from file2
    with jsonlines.open(file2, "r") as reader2:
        for obj in reader2:
            writer.write(obj)

print("✓ Merged into", output)

✓ Merged into ea_items_merged.jsonl


In [ ]:
# Install required packages
%pip install transformers accelerate peft bitsandbytes datasets torch jsonlines -q
# # for qwen-chat-7b
# !pip install transformers==4.37.2 accelerate bitsandbytes==0.43.1 peft==0.11.1 --force-reinstall
# !pip install transformers_stream_generator


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 46.4 MB/s eta 0:00:00


In [ ]:
# !pip install trl torch jsonlines -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.9.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.
torchvision 0.24.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.


In [ ]:
import transformers
transformers.__version__

'4.57.3'

In [ ]:
import json
import jsonlines
import os
from typing import Dict, List, Optional
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
from tqdm import tqdm

print("✓ Libraries imported")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")


✓ Libraries imported
✓ PyTorch version: 2.9.0+cu126
✓ CUDA available: True
✓ GPU: NVIDIA A100-SXM4-40GB


## Data Loading and Formatting

Convert EU and EA examples to instruction-response pairs for fine-tuning.


In [ ]:
# Configuration
INPUT_DIR = "/content/drive/MyDrive/685_Project/input_data/finetuning/"  # Adjust if your data is elsewhere
EU_FILE = os.path.join(INPUT_DIR, "eu_items.jsonl")
EA_FILE = os.path.join(INPUT_DIR, "ea_items.jsonl")
EU_EA_FILE=os.path.join(INPUT_DIR, "eu_ea_dataset_items_final.jsonl")

# Check if files exist
# if not os.path.exists(EU_FILE):
#     print(f"⚠ EU file not found: {EU_FILE}")
#     print("Please update DATA_DIR or ensure generated data exists")
# if not os.path.exists(EA_FILE):
#     print(f"⚠ EA file not found: {EA_FILE}")
#     print("Please update DATA_DIR or ensure generated data exists")
if not os.path.exists(EU_EA_FILE):
    print(f"⚠ Input file not found: {EU_EA_FILE}")
    print("Please update DATA_DIR or ensure generated data exists")

print(f"Data directory: {INPUT_DIR}")
# print(f"EU file: {EU_FILE}")
# print(f"EA file: {EA_FILE}")


Data directory: /content/drive/MyDrive/685_Project/input_data/finetuning/


In [ ]:
import json
import jsonlines
import pandas as pd
from sklearn.model_selection import train_test_split


# ================================
# 1. SAFE JSONL LOADING (robust)
# ================================
def load_jsonl_safely(path):
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                items.append(obj)
            except json.JSONDecodeError:
                # Try to manually fix trailing commas or escape errors
                try:
                    obj = json.loads(line.rstrip(","))
                    items.append(obj)
                except Exception:
                    print(f"⚠️ Skipping malformed line:\n{line[:200]}")
    return items


# ==============================================
# 2. FORMATTERS (CLEAN & QUOTE-SAFE VERSIONS)
# ==============================================

def format_eu_example(eu_item):
    scenario = eu_item.get("scenario", "")
    emotion_choices = eu_item.get("emotion_choices", [])
    cause_choices = eu_item.get("cause_choices", [])
    emotion_label = eu_item.get("emotion_label", "")
    cause_label = eu_item.get("cause_label", "")

    # Map emotion label → A/B/C/D…
    emotion_index = None
    for i, choice in enumerate(emotion_choices):
        if choice == emotion_label:
            emotion_index = chr(65 + i)
            break

    # Map cause label → 1/2/3/4…
    cause_index = None
    for i, choice in enumerate(cause_choices):
        if choice == cause_label:
            cause_index = str(i + 1)
            break

    # Build instruction
    instruction = (
        "You are an emotionally intelligent assistant.\n\n"
        "Read the scenario and answer two questions:\n"
        "1. What emotions is the person most likely feeling?\n"
        "2. What are the main causes of these emotions?\n\n"
        f"Scenario: {scenario}\n\n"
        "Options for emotions:\n"
    )

    for i, choice in enumerate(emotion_choices):
        instruction += f"{chr(65+i)}. {choice}\n"

    instruction += "\nOptions for causes:\n"
    for i, choice in enumerate(cause_choices):
        instruction += f"{i+1}. {choice}\n"

    instruction += (
        "\nThink step by step, then answer:\n"
        "Emotions: <letter>\nCauses: <number>"
    )

    # Response
    response = (
        f"Reasoning: The person is feeling {emotion_label} because {cause_label}.\n\n"
        f"Emotions: {emotion_index}\nCauses: {cause_index}"
    )

    return {
        "input": instruction,
        "output": response,
        "type": "EU"
    }


def format_ea_example(ea_item):
    scenario = ea_item.get("scenario", "")
    choices = ea_item.get("choices", [])
    label = ea_item.get("label", "")

    # Clean label
    clean_label = label.strip('"').strip("'")

    # Find correct index
    choice_index = None
    for i, choice in enumerate(choices):
        clean_choice = choice.strip('"').strip("'")
        if clean_choice == clean_label:
            choice_index = chr(65 + i)
            break

    # Instruction
    instruction = (
        "You are an emotionally intelligent assistant.\n\n"
        "Read the situation and choose the most effective response.\n\n"
        f"Scenario: {scenario}\n\nOptions:\n"
    )

    for i, choice in enumerate(choices):
        clean_choice = choice.strip('"').strip("'")
        instruction += f"{chr(65+i)}. {clean_choice}\n"

    instruction += (
        "\nThink step by step, then answer:\n"
        "Response: <letter>"
    )

    response = (
        f"Reasoning: Option {choice_index} best addresses the person's emotional needs.\n\n"
        f"Response: {choice_index}"
    )

    return {
        "input": instruction,
        "output": response,
        "type": "EA"
    }


def format_unified_example(item):
    task_type = item.get("task_type", "").upper()
    scenario = item.get("scenario", "")
    question = item.get("question", "")
    choices_dict = item.get("choices", {})
    answer = item.get("answer", "")
    rationale = item.get("rationale", "")

    choice_keys = sorted(choices_dict.keys())

    # Instruction header
    if task_type == "EU":
        task_text = "Task: Emotional Understanding"
    else:
        task_text = "Task: Emotional Application"

    instruction = (
        "You are an emotionally intelligent assistant.\n\n"
        f"{task_text}\n\n"
        f"Scenario: {scenario}\n\n"
        f"Question: {question}\n\nOptions:\n"
    )

    for key in choice_keys:
        instruction += f"{key}. {choices_dict[key]}\n"

    instruction += (
        "\nThink step by step.\nAnswer: <letter>"
    )

    response = f"Reasoning: {rationale}\n\nAnswer: {answer}"

    return {
        "input": instruction,
        "output": response,
        "type": task_type
    }


# ==================================
# 3. LOAD ALL 3 DATA FILES CLEANLY
# ==================================

INPUT_DIR = "/content/drive/MyDrive/685_Project_New/input_data/finetuning/Laukik/"  # modify if needed

eu_items_raw = load_jsonl_safely(INPUT_DIR + "eu_items-3.jsonl")
ea_items_raw = load_jsonl_safely(INPUT_DIR + "ea_items-3.jsonl")
# unified_items_raw = load_jsonl_safely(INPUT_DIR + "arjuns_generation.jsonl")

print("Loaded counts:")
print("EU:", len(eu_items_raw))
print("EA:", len(ea_items_raw))
# print("Unified:", len(unified_items_raw))


# =====================================
# 4. FORMAT EVERYTHING UNIFORMLY
# =====================================

formatted = []

for item in eu_items_raw:
    formatted.append(format_eu_example(item))

for item in ea_items_raw:
    formatted.append(format_ea_example(item))

# for item in unified_items_raw:
#     formatted.append(format_unified_example(item))

df = pd.DataFrame(formatted)
print("\nTotal formatted examples:", len(df))


# =======================================================
# 5. STRATIFIED SPLIT ENSURING SAME EU/EA PROPORTIONS
# =======================================================

train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["type"]  # ensures EU/EA ratio preserved
)

print("Train:", len(train_df), "| Val:", len(val_df))


# ===================================
# 6. SAVE TRAIN/VAL DATA FOR FINETUNE
# ===================================

import os
os.makedirs("training_data", exist_ok=True)

with jsonlines.open("training_data/train.jsonl", "w") as w:
    for _, row in train_df.iterrows():
        w.write({"input": row["input"], "output": row["output"]})

with jsonlines.open("training_data/val.jsonl", "w") as w:
    for _, row in val_df.iterrows():
        w.write({"input": row["input"], "output": row["output"]})

print("\n✓ Saved training files:")
print("training_data/train.jsonl")
print("training_data/val.jsonl")

Loaded counts:
EU: 639
EA: 988

Total formatted examples: 1627
Train: 1301 | Val: 326

✓ Saved training files:
training_data/train.jsonl
training_data/val.jsonl


## Model Setup: GPT-OSS-20B with QLoRA


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model

# -------------------------------
# UNIVERSAL MODEL LOADER + QLORA
# -------------------------------

def load_trainable_model(
    model_name,
    r=64,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
):
    print(f"\n🔍 Loading model: {model_name}")

    # -------------------------------
    # 1. Load tokenizer (always safe)
    # -------------------------------
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # -------------------------------
    # 2. Try loading model with 4-bit QLoRA
    # -------------------------------
    print("🧪 Checking if BitsAndBytes 4-bit is supported...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        print("✅ Loaded with 4-bit NF4 QLoRA")
        quantization_type = "bnb_4bit"

    except ValueError as e:
        if "mxfp4" in str(e).lower():
            print("❌ Model is MXFP4 → Inference-only → Cannot fine-tune.")
            print("   Choose another model (e.g., Mistral-7B, Gemma-2B, Qwen2.5-7B).")
            return None, tokenizer

        elif "awq" in str(e).lower():
            print("❌ Model is AWQ quantized → LoRA not supported.")
            return None, tokenizer

        else:
            print("⚠️ Unknown quantization error:", e)
            return None, tokenizer

    except Exception as e:
        print("❌ Failed to load model:", e)
        return None, tokenizer

    # -------------------------------
    # 3. Attach LoRA adapters
    # -------------------------------
    print("🛠️ Attaching LoRA adapters...")

    lora_config = LoraConfig(
        r=r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=target_modules,
    )

    try:
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()
        model.to("cuda")
        print("✅ LoRA attached successfully!")
    except Exception as e:
        print("❌ Failed to attach LoRA:", e)
        print("   Likely the model architecture does not expose q/k/v/o projections.")
        return None, tokenizer

    print("\n🎉 Model is ready for QLoRA finetuning!")
    print("--------------------------------------------------------")

    return model, tokenizer

model,tokenizer=load_trainable_model(model_name="Qwen/Qwen2.5-7B-Instruct")


🔍 Loading model: Qwen/Qwen2.5-7B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


🧪 Checking if BitsAndBytes 4-bit is supported...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Loaded with 4-bit NF4 QLoRA
🛠️ Attaching LoRA adapters...
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
✅ LoRA attached successfully!

🎉 Model is ready for QLoRA finetuning!
--------------------------------------------------------


## Dataset Preparation


In [ ]:
# Load dataset from JSONL files
train_file="training_data/train.jsonl"
val_file="training_data/val.jsonl"
dataset = load_dataset(
    "json",
    data_files={
        "train": train_file,
        "val": val_file
    }
)

print(f"✓ Dataset loaded")
print(f"  - Train: {len(dataset['train'])} examples")
print(f"  - Val: {len(dataset['val'])} examples")


Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

✓ Dataset loaded
  - Train: 1301 examples
  - Val: 326 examples


In [ ]:

# Combine input and output into a single text sequence
def format_example(example):
    """
    Format instruction-response pair for causal LM training.
    """
    # Combine input and output
    text = example["input"] + "\n\nAssistant:\n" + example["output"] + tokenizer.eos_token

    # Tokenize
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=1024,  # Adjust based on your needs
        padding="max_length"
    )

    # Labels are the same as input_ids (we'll mask the prompt part if needed)
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

# Apply formatting
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    format_example,
    remove_columns=dataset["train"].column_names,
    batched=False
)

print("✓ Dataset tokenized")
print(f"  - Train: {len(tokenized_dataset['train'])} examples")
print(f"  - Val: {len(tokenized_dataset['val'])} examples")

# Show sample
if len(tokenized_dataset['train']) > 0:
    sample = tokenized_dataset['train'][0]
    print(f"\nSample tokenized example:")
    print(f"  - Input IDs length: {len(sample['input_ids'])}")
    print(f"  - Labels length: {len(sample['labels'])}")
    print(f"  - Decoded text preview:")
    decoded = tokenizer.decode(sample['input_ids'][:100], skip_special_tokens=False)
    print(f"    {decoded[:200]}...")


Tokenizing dataset...


Map:   0%|          | 0/1301 [00:00<?, ? examples/s]

Map:   0%|          | 0/326 [00:00<?, ? examples/s]

✓ Dataset tokenized
  - Train: 1301 examples
  - Val: 326 examples

Sample tokenized example:
  - Input IDs length: 1024
  - Labels length: 1024
  - Decoded text preview:
    You are an emotionally intelligent assistant.

Read the situation and choose the most effective response.

Scenario: I want to help my local community adapt to AI fire risk predictions, but I’m unsure...


## Training Configuration


In [ ]:
# Training arguments
from transformers import TrainingArguments
from transformers import TrainingArguments, EarlyStoppingCallback
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/685_Project/fine-tuned_model/Laukik/Qwen/Finetuned-Qwen2.5-7B-Instruct",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=20,
    save_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    report_to="none",
    warmup_steps=100,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
)

print("✓ Training arguments configured")
print(f"  - Output dir: {training_args.output_dir}")
print(f"  - Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  - Learning rate: {training_args.learning_rate}")
print(f"  - Epochs: {training_args.num_train_epochs}")


✓ Training arguments configured
  - Output dir: /content/drive/MyDrive/685_Project/fine-tuned_model/Laukik/Qwen/Finetuned-Qwen2.5-7B-Instruct
  - Effective batch size: 32
  - Learning rate: 0.0002
  - Epochs: 3


In [ ]:
from transformers import EarlyStoppingCallback

# --- Data collator ---
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # causal LM
)


# --- Early stopping callback ---
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3,      # stop after 3 evals with no improvement
    early_stopping_threshold=0.0    # require strictly better eval_loss
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["val"],
    data_collator=data_collator,
)

## Start Training

⚠️ **Note**: Training will take several hours depending on dataset size and hardware. Monitor GPU memory usage.


In [ ]:
len(train_examples)

610

In [ ]:
len(tokenized_dataset["val"])

153

In [ ]:
# Start training
print("Starting training...")
print("This may take several hours. Monitor the logs for progress.")

trainer.train()

print("\n✓ Training complete!")

Starting training...
This may take several hours. Monitor the logs for progress.


Step,Training Loss,Validation Loss
50,2.131300,1.132923
100,0.775300,0.768338



✓ Training complete!


In [ ]:
# Save the final model
final_model_dir = "/content/drive/MyDrive/685_Project_New/fine-tuned_model/Laukik/Qwen/Finetuned-Qwen2.5-7B-Instruct"
model.save_pretrained(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

print(f"✓ Model saved to: {final_model_dir}")
print("You can load it later with:")
print(f"  from peft import PeftModel")
print(f"  model = PeftModel.from_pretrained(base_model, '{final_model_dir}')")


✓ Model saved to: /content/drive/MyDrive/685_Project_New/fine-tuned_model/Laukik/Qwen/Finetuned-Qwen2.5-7B-Instruct
You can load it later with:
  from peft import PeftModel
  model = PeftModel.from_pretrained(base_model, '/content/drive/MyDrive/685_Project_New/fine-tuned_model/Laukik/Qwen/Finetuned-Qwen2.5-7B-Instruct')


In [ ]:
!git clone https://github.com/kernelism/EmoBenchModified.git

fatal: destination path 'EmoBenchModified' already exists and is not an empty directory.


In [ ]:
!python -m pip install langchain_core==0.3.60 langchain_openai==0.3.17 json_repair

In [ ]:
%cd EmoBenchModified

/content/EmoBenchModified


In [ ]:
!python /content/EmoBenchModified/src/main.py \
  --model_type "HF" \
  --model_path "Qwen/Qwen2.5-7B-Instruct" \
  --lang "en" \
  --task "all" \
  --device 0

2025-12-15 22:14:35.811459: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-15 22:14:35.828676: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765836875.850294    1383 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765836875.856679    1383 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765836875.872906    1383 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!python /content/EmoBenchModified/src/main.py \
  --model_type "HF" \
  --model_path "/content/drive/MyDrive/685_Project_New/fine-tuned_model/Laukik/Qwen/Finetuned-Qwen2.5-7B-Instruct" \
  --lang "en" \
  --task "all" \
  --device 0


2025-12-18 04:17:56.414881: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-18 04:17:56.433456: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766031476.455210    2476 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766031476.461794    2476 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766031476.478883    2476 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Evaluation on Real EmoBench Data

Evaluate the fine-tuned model on real EmoBench EU and EA questions.


In [ ]:
# # Load evaluation data (real EmoBench examples)
# # You'll need to provide the path to real EmoBench data
# EMOBENCH_EU_FILE = "/content/drive/MyDrive/685_Project/emobench_data/EU.jsonl"  # Update with actual path
# EMOBENCH_EA_FILE = "/content/drive/MyDrive/685_Project/emobench_data/EA.jsonl"  # Update with actual path

# def evaluate_eu_item(model, tokenizer, eu_item: Dict) -> Dict:
#     """
#     Evaluate model on a single EU item.
#     Returns prediction and correctness.
#     """
#     # Format as instruction
#     scenario = eu_item.get("scenario", "")
#     emotion_choices = eu_item.get("emotion_choices", [])
#     cause_choices = eu_item.get("cause_choices", [])
#     emotion_label = eu_item.get("emotion_label", "")
#     cause_label = eu_item.get("cause_label", "")

#     instruction = f"""You are an emotionally intelligent assistant.

# Read the scenario and answer two questions:
# 1. What emotions is the person most likely feeling?
# 2. What are the main causes of these emotions?

# Task: Emotional Understanding

# Scenario: {scenario}

# Options for emotions:
# """
#     for i, choice in enumerate(emotion_choices):
#         instruction += f"{chr(65 + i)}. {choice}\n"

#     instruction += "\nOptions for causes:\n"
#     for i, choice in enumerate(cause_choices):
#         instruction += f"{i + 1}. {choice}\n"

#     instruction += "\nThink step by step about the situation, then answer with:\nEmotions: <letter>\nCauses: <number>"

#     # Generate response
#     inputs = tokenizer(instruction, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=200,
#             temperature=0.7,
#             do_sample=True,
#             pad_token_id=tokenizer.pad_token_id,
#             eos_token_id=tokenizer.eos_token_id
#         )

#     response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

#     # Parse response to extract emotion and cause predictions
#     # Look for "Emotions: X" and "Causes: Y" patterns
#     import re
#     emotion_match = re.search(r'Emotions:\s*([A-F])', response, re.IGNORECASE)
#     cause_match = re.search(r'Causes:\s*(\d+)', response, re.IGNORECASE)

#     pred_emotion_idx = emotion_match.group(1) if emotion_match else None
#     pred_cause_idx = cause_match.group(1) if cause_match else None

#     # Find correct indices
#     true_emotion_idx = None
#     for i, choice in enumerate(emotion_choices):
#         if choice == emotion_label:
#             true_emotion_idx = chr(65 + i)
#             break

#     true_cause_idx = None
#     for i, choice in enumerate(cause_choices):
#         if choice == cause_label:
#             true_cause_idx = str(i + 1)
#             break

#     emotion_correct = (pred_emotion_idx == true_emotion_idx) if pred_emotion_idx and true_emotion_idx else False
#     cause_correct = (pred_cause_idx == true_cause_idx) if pred_cause_idx and true_cause_idx else False

#     return {
#         "qid": eu_item.get("qid", ""),
#         "pred_emotion": pred_emotion_idx,
#         "true_emotion": true_emotion_idx,
#         "emotion_correct": emotion_correct,
#         "pred_cause": pred_cause_idx,
#         "true_cause": true_cause_idx,
#         "cause_correct": cause_correct,
#         "both_correct": emotion_correct and cause_correct,
#         "response": response
#     }


# def evaluate_ea_item(model, tokenizer, ea_item: Dict) -> Dict:
#     """
#     Evaluate model on a single EA item.
#     Returns prediction and correctness.
#     """
#     scenario = ea_item.get("scenario", "")
#     choices = ea_item.get("choices", [])
#     label = ea_item.get("label", "")

#     instruction = f"""You are an emotionally intelligent assistant.

# Read the situation and choose the most effective, validating response.

# Task: Emotional Application

# Scenario: {scenario}

# Options:
# """
#     for i, choice in enumerate(choices):
#         clean_choice = choice.strip('"').strip("'")
#         instruction += f"{chr(65 + i)}. {clean_choice}\n"

#     instruction += "\nThink step by step about how each option would make the person feel.\nThen answer with the single best option as:\nResponse: <letter>"

#     # Generate response
#     inputs = tokenizer(instruction, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=150,
#             temperature=0.7,
#             do_sample=True,
#             pad_token_id=tokenizer.pad_token_id,
#             eos_token_id=tokenizer.eos_token_id
#         )

#     response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

#     # Parse response
#     import re
#     response_match = re.search(r'Response:\s*([A-D])', response, re.IGNORECASE)
#     pred_idx = response_match.group(1) if response_match else None

#     # Find correct index
#     true_idx = None
#     clean_label = label.strip('"').strip("'")
#     for i, choice in enumerate(choices):
#         clean_choice = choice.strip('"').strip("'")
#         if clean_choice == clean_label or choice == label:
#             true_idx = chr(65 + i)
#             break

#     correct = (pred_idx == true_idx) if pred_idx and true_idx else False

#     return {
#         "qid": ea_item.get("qid", ""),
#         "pred": pred_idx,
#         "true": true_idx,
#         "correct": correct,
#         "response": response
#     }

# print("✓ Evaluation functions defined")


✓ Evaluation functions defined


In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from peft import PeftModel
# import torch

# # Load base model
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
# )

# base_model = AutoModelForCausalLM.from_pretrained(
#     "meta-llama/Llama-3.1-8B",
#     quantization_config=bnb_config,
#     device_map="auto",
#     trust_remote_code=True
# )

# # Load LoRA adapters
# model = PeftModel.from_pretrained(base_model, "/content/drive/MyDrive/685_Project/fine-tuned_model/llama_emobench_lora_final")
# tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/685_Project/fine-tuned_model/llama_emobench_lora_final")

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

The tokenizer you are loading from '/content/drive/MyDrive/685_Project/fine-tuned_model/llama_emobench_lora_final' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e.  This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
